In [37]:
#STEP 1: Environment setup and AuraDB credentials loading
import nest_asyncio
nest_asyncio.apply()

import warnings
warnings.filterwarnings("ignore")

import os

# Manually parse .env file 
with open(".env", "r") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

uri = os.environ.get("NEO4J_URI")
username = os.environ.get("NEO4J_USERNAME")
password = os.environ.get("NEO4J_PASSWORD")
database = os.environ.get("NEO4J_DATABASE")

print("URI loaded:", uri is not None)
print("Username loaded:", username is not None)
print("Password loaded:", password is not None)
print("Database loaded:", database is not None)

URI loaded: True
Username loaded: True
Password loaded: True
Database loaded: True


In [38]:
#STEP 2: NEO4j AuraDB CONNECTION TEST
# Run this to confirm credentials are working before running the full pipeline
from neo4j import GraphDatabase
driver = GraphDatabase.driver(uri, auth=(username, password))

try:
    with driver.session() as session:
        result = session.run("RETURN 'Connected to Neo4j AuraDB!' AS message")
        for record in result:
            print(record["message"])
    print("Connection successful!")
except Exception as e:
    print("Connection failed:", e)
finally:
    driver.close()

Connected to Neo4j AuraDB!
Connection successful!


In [39]:
#STEP 3: ChatGPT TEST
from llama_index.llms.openai import OpenAI
import os

llm = OpenAI(
    model="gpt-4o-mini",
    api_key=os.environ.get("OPENAI_API_KEY"),
    temperature=0,
    timeout=120.0,
)

response = llm.complete("Say hello and confirm you are running via the OpenAI API.")
print(response)
print(response.raw)  # shows token usage

Hello! I can confirm that I am running via the OpenAI API. How can I assist you today?
ChatCompletion(id='chatcmpl-E2ClTK44bx27N3yaeNiJiK0QQARNh', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! I can confirm that I am running via the OpenAI API. How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1784194431, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_a8eb499e0d', usage=CompletionUsage(completion_tokens=22, prompt_tokens=20, total_tokens=42, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [ ]:
#STEP 3: OLLAMA TEST after installing Ollama locally from https://ollama.com
#from llama_index.llms.ollama import Ollama

#llm = Ollama(model="llama3:8b", request_timeout=120.0)

#response = llm.complete("Say hello and confirm you are running locally.")
#print(response)

Hello!

Just to confirm, I'm running locally on your device, which means I don't have any external dependencies or connections. I'm a self-contained AI model designed to assist with tasks and answer questions to the best of my abilities. How can I help you today?


In [40]:
# STEP 4: PDF text EXTRACTION from the module
import pdfplumber
import os
from docx import Document as DocxDocument

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
    return text

def extract_text_from_docx(docx_path):
    doc = DocxDocument(docx_path)
    text = ""
    for paragraph in doc.paragraphs:
        if paragraph.text.strip():
            text += paragraph.text + "\n"
    return text

pdf_files = [f for f in os.listdir("data") if f.endswith(".pdf")]
docx_files = [f for f in os.listdir("data") if f.endswith(".docx")]

print(f"PDF files found: {pdf_files}")
print(f"DOCX files found: {len(docx_files)}")

# Test PDF extraction
if pdf_files:
    text = extract_text_from_pdf(f"data/{pdf_files[0]}")
    print(f"\nPDF preview: {text[:200]}")

# Test DOCX extraction
if docx_files:
    text = extract_text_from_docx(f"data/{docx_files[0]}")
    print(f"\nDOCX preview: {text[:200]}")


PDF files found: ['MathsComp_Lecture1.pdf', 'MathsComp_Lecture10.pdf', 'MathsComp_Lecture11.pdf', 'SeminarTasks12_MathComp.pdf', 'SeminarTasks3_MathComp.pdf', 'SeminarTasks4_MathComp.pdf', 'MathsComp_Lecture2.pdf', 'MathComp-Module-Proforma-Sep2024.pdf', 'MathsComp_Lecture3.pdf', 'MathsComp_Lecture7.pdf', 'MathsComp_Lecture4.pdf', 'MathsComp_Lecture5.pdf', 'SeminarTasks2_MathComp.pdf', 'ICT-Revision-Session-November2025.pdf', 'SeminarTasks9_MathComp.pdf', 'MathsComp-Q&ASession7.pdf', 'SeminarTasks11_MathComp.pdf', 'MathsComp_Lecture8.pdf', 'MathsComp_Lecture9.pdf', 'MathComp Schedule SEP2025 StudentsVersion.pdf', 'SeminarTasks0_MathComp.pdf', 'SeminarTasks7_MathComp.pdf', 'SeminarTasks10_MathComp.pdf', 'SeminarTasks1_MathComp.pdf']
DOCX files found: 10

PDF preview: 4COSC002W Mathematics
for Computing
Lecture 1
Module Introduction. Number Formats. Types of Numbers.
Number Theory Basics. Modular Arithmetic. Sequences.
WELCOME
TO THE MATHS FOR COMPUTING MODULE!
Mod

DOCX preview: 4COSCOO

In [41]:
#STEP 5: Check which documents are already processed in AuraDB
from neo4j import GraphDatabase

driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session(database=database) as session:
    result = session.run("""
        MATCH (p:ProcessedDocument) 
        RETURN p.filename AS filename
    """)
    processed_files = {record["filename"] for record in result}

driver.close()

print(f"Documents already in AuraDB: {len(processed_files)}")
for f in sorted(processed_files):
    print(f"  ✓ {f}")

Documents already in AuraDB: 0


In [42]:
#STEP 6:Scan data folder and identify new documents not yet processed
import os
import re

# Pipeline scope: lecture PDF files only.
# Seminar tasks, module proforma, schedules, and revision sessions
# are excluded from this phase. Extending to DOCX seminar files
# is identified as future work.

all_files = os.listdir("data")
new_files = []

for filename in all_files:
    # Only process PDF files whose name contains "Lecture"
    if not filename.endswith(".pdf"):
        continue
    if not re.search(r"[Ll]ecture", filename):
        continue
    if filename in processed_files:
        print(f"⏭ Already processed: {filename}")
    else:
        new_files.append(filename)
        print(f"🆕 New document found: {filename}")

print(f"\nTotal new documents to process: {len(new_files)}")

🆕 New document found: MathsComp_Lecture1.pdf
🆕 New document found: MathsComp_Lecture10.pdf
🆕 New document found: MathsComp_Lecture11.pdf
🆕 New document found: MathsComp_Lecture2.pdf
🆕 New document found: MathsComp_Lecture3.pdf
🆕 New document found: MathsComp_Lecture7.pdf
🆕 New document found: MathsComp_Lecture4.pdf
🆕 New document found: MathsComp_Lecture5.pdf
🆕 New document found: MathsComp_Lecture8.pdf
🆕 New document found: MathsComp_Lecture9.pdf

Total new documents to process: 10


In [43]:
#STEP 7: PIPELINE SETUP
import re
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor
from typing import Literal
from llama_index.core import Settings
from llama_index.core.callbacks import CallbackManager, TokenCountingHandler
import tiktoken

#llm = Ollama(model="llama3:8b", request_timeout=3000.0, temperature=0)
#llm = Ollama(model="phi3:14b", request_timeout=1600.0, temperature=0)
#llm = Ollama(model="phi3-mini-4k", request_timeout=600.0, temperature=0)



token_counter = TokenCountingHandler(
    tokenizer=tiktoken.encoding_for_model("gpt-4o-mini").encode
)
callback_manager = CallbackManager([token_counter])

llm = OpenAI(
    model="gpt-4o-mini",
    api_key=os.environ.get("OPENAI_API_KEY"),
    temperature=0,
    timeout=600.0,
    additional_kwargs={"seed": 42},
    callback_manager=callback_manager,
)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")


Settings.callback_manager = callback_manager



# TEXT PREPROCESSING
# Most patterns are generic across slide-based modules.
# The final two patterns are specific to the Maths for Computing module
# and should be reviewed when applying the pipeline to a different module.
SKIP_SLIDE_PATTERNS = [
    r"^WELCOME",
    r"^Module Team",
    r"^Sources\s*$",
    r"^Independent Study",
    r"^Lectures\s*$",
    r"^Seminars\s*$",
    r"^Q&A SESSION",
    r"^ACTIVITY\s*\d*[:\.]?\s*$",
    r"^HOMEWORK ACTIVITY",
    r"^Let'?s get closer",
    r"^Module Organisation\s*$",
    r"^Module Syllabus\s*$",
    r"^Discrete Mathematics\s*$",
    r"^Mathematics\s+associated\s+with",
]

NOISE_LINE_PATTERNS = [
    r"^(Property|Attribute|Definition|Notation|Ordering|Repetition|"
    r"Membership|Indexing|Cardinality|Length|Inclusivity|Elements?|"
    r"Representation|Type|Notes?|Description|Example)\s*$",
    r"^.{3,40}\s+(Examples?|Notation|Properties|Representation|Types?|Basics)\s*$",
    r"^\s*[|─┼+\-]{3,}",
    r"^\s*[∅∈∉⊆⊂∪∩×′]\s*$",
]

def preprocess_text(raw_text: str) -> str:
    text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = re.split(r"\n{2,}", text)
    cleaned_blocks = []
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        # Check first 5 lines for skip patterns — headers/footers mean
        # slide title may not always be the very first line
        first_lines = [l.strip() for l in block.split("\n")[:5]]
        if any(
            re.match(p, line, re.IGNORECASE)
            for p in SKIP_SLIDE_PATTERNS
            for line in first_lines
        ):
            continue
        lines = block.split("\n")
        clean_lines = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            if any(re.match(p, line, re.IGNORECASE) for p in NOISE_LINE_PATTERNS):
                continue
            if len(line) < 4 and not re.search(r"[a-zA-Z]{2,}", line):
                continue
            clean_lines.append(line)
        if len(clean_lines) >= 3:
            cleaned_blocks.append("\n".join(clean_lines))
    return "\n\n".join(cleaned_blocks)

# EXTRACTION SCHEMA
# Entity types and relationships defined jointly with O4 ontology (ontology_v1.owl)
# TOPIC list is authoritative for module 4COSC002W — update when ontology is revised
entities = Literal[
    "CONCEPT",      # named mathematical idea e.g. Modular Arithmetic
    "TOPIC",        # thematic cluster e.g. Number Theory
    "WEEK",         # source week anchor e.g. Week 1
    "APPLICATION",  # real-world domain e.g. Cryptography
    "ASSESSMENT",   # ICT, Exam, Mock Test
    "DEFINITION",   # formal definition of a concept
    "EXAMPLE",      # concrete worked example illustrating a concept
    "FORMULA",      # mathematical formula or equation
]

relations = Literal[
    "INTRODUCED_IN",     # concept first introduced in a week
    "PART_OF",           # concept belongs to topic or week; topic belongs to week
    "PREREQUISITE_OF",   # must understand A before B
    "HAS_DEFINITION",    # concept has a formal definition
    "HAS_EXAMPLE",       # concept has a worked example
    "HAS_FORMULA",       # concept has an associated formula
    "HAS_APPLICATION",   # concept applies to a real-world domain
    "ASSESSED_BY",       # concept appears in an assessment
    "USED_IN",           # concept is used within another concept
]

validation_schema = [
    ("CONCEPT", "INTRODUCED_IN", "WEEK"),
    ("CONCEPT", "PART_OF", "TOPIC"),
    ("CONCEPT", "PART_OF", "WEEK"),
    ("TOPIC",   "PART_OF", "WEEK"),
    ("CONCEPT", "PREREQUISITE_OF", "CONCEPT"),
    ("CONCEPT", "HAS_DEFINITION", "DEFINITION"),
    ("CONCEPT", "HAS_EXAMPLE", "EXAMPLE"),
    ("CONCEPT", "HAS_FORMULA", "FORMULA"),
    ("CONCEPT", "HAS_APPLICATION", "APPLICATION"),
    ("TOPIC",   "HAS_APPLICATION", "APPLICATION"),
    ("CONCEPT", "ASSESSED_BY", "ASSESSMENT"),
    ("CONCEPT", "USED_IN", "CONCEPT"),
]

extraction_prompt = """
Extract a knowledge graph from the following educational text from a \
Mathematics for Computing university module.

ENTITY RULES:
- CONCEPT: A specific named mathematical term actively taught in this text.
  Good examples by topic:
  Number Theory: "Prime Numbers", "Modular Arithmetic", "Division Theorem", "Fibonacci Sequence"
  Set Theory: "Cartesian Product", "Power Set", "Set Builder Notation", "Empty Set"
  Logics: "Proposition", "Truth Table", "Logical Operator", "Tautology", "Predicate"
  Relations: "Binary Relation", "Reflexive", "Symmetric", "Equivalence Relation"
  Functions: "Injective", "Surjective", "Bijective", "Composite Function"
  Graph Theory: "Vertex", "Edge", "Adjacency Matrix", "Tree", "Path"
  Matrices: "Matrix Multiplication", "Determinant", "Inverse Matrix", "Transpose"
  Probability: "Random Variable", "Probability Distribution", "Bayes Theorem"
  IMPORTANT: Named mathematical terms like number types (Integer, Rational Number,
  Prime Number, Natural Number) and named theorems (Division Theorem, Fundamental
  Theorem of Arithmetic) are ALWAYS CONCEPT nodes even when the text provides
  their definition or formula. Extract the definition separately as a DEFINITION
  node linked via HAS_DEFINITION — do not replace the CONCEPT with a DEFINITION.

- TOPIC: A major mathematical theme. You MUST only use names from this exact list:
  "Number Theory", "Sequences", "Set Theory", "Intervals", "Relations",
  "Functions", "Logics", "Graph Theory", "Matrices Part 1", "Matrices Part 2",
  "Probability", "Statistics Basics", "Statistical Application"
  Do NOT invent new TOPIC names. Do NOT create TOPICs for subjects only mentioned
  in passing.

- WEEK: Use only the week identifier from the source (e.g. "Week 1", "Week 2").
  NEVER classify a slide heading as a WEEK node.
  NEVER extract "Recordings", "Lecture Notes", "Seminar" as nodes.

- APPLICATION: A real-world domain explicitly connected to a concept in the text.
  Good: "Cryptography", "Computer Graphics", "Hash Functions"
  Only extract if the text explicitly states this concept is used in this domain.

- ASSESSMENT: Only "ICT", "Exam", or "Mock Test". No other values.

- DEFINITION: The formal or textbook definition of a concept, expressed as a
  short complete sentence or phrase. One definition per concept maximum.
  Good: "A prime number is a natural number greater than 1 with no positive
  divisors other than 1 and itself."

- EXAMPLE: A concrete numerical or worked example that illustrates a concept.
  Good: "17 mod 5 = 2", "The set {1, 2, 3} has cardinality 3"
  Only extract if the text gives a specific worked instance, not a general statement.

- FORMULA: A mathematical formula or symbolic expression associated with a concept.
  Good: "a ≡ b (mod n)", "n! = n × (n-1) × ... × 1"
  Only extract standalone symbolic expressions, not prose descriptions.

STRICT EXCLUSION RULES:
- NEVER extract table column headers as concepts
- NEVER extract slide sub-headings ending in "Examples", "Basics", "Notation",
  "Properties", "Representation", "Types" as concept nodes
- NEVER extract numeric values or lone symbols as concepts
- NEVER extract parenthetical variants: use "Integers" not "Integers (Z)"

Text: {text}
"""

kg_extractor = SchemaLLMPathExtractor(
    llm=llm,
    possible_entities=entities,
    possible_relations=relations,
    kg_validation_schema=validation_schema,
    extract_prompt=extraction_prompt,
    strict=True
)

print("Pipeline components ready:")
print(f"✓ LLM: {llm.model} (local)")
print("✓ Embeddings: BAAI/bge-small-en-v1.5 (local)")
print("✓ Schema: defined with", len(validation_schema), "relationship types")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6655.77it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline components ready:
✓ LLM: gpt-4o-mini (local)
✓ Embeddings: BAAI/bge-small-en-v1.5 (local)
✓ Schema: defined with 12 relationship types


In [36]:
# OPTIONAL!!! Run this to clear AuraDB before rebuilding
# WARNING!!! This deletes everything in the database

from neo4j import GraphDatabase

driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session(database=database) as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("Database cleared — ready for fresh build")
driver.close()

Database cleared — ready for fresh build


In [44]:
# STEP 8: Connect to existing AuraDB graph or initialise fresh
from llama_index.core import PropertyGraphIndex
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore

graph_store = Neo4jPropertyGraphStore(
    username=username,
    password=password,
    url=uri,
    database=database
)

# Check if graph already exists
driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session(database=database) as session:
    count = session.run("MATCH (n) RETURN count(n) AS count").single()["count"]
driver.close()

if count > 0:
    # Graph already exists — connect to it without rebuilding
    print(f"Existing graph found: {count} nodes in AuraDB")
    print("Connecting to existing graph...")
    index = PropertyGraphIndex.from_existing(
        property_graph_store=graph_store,
        llm=llm,
        embed_model=embed_model,
    )
    print("Connected to existing graph successfully!")
else:
    # No graph exists — build fresh
    print("No existing graph found — initialising fresh index...")
    index = PropertyGraphIndex.from_documents(
        [],
        kg_extractors=[kg_extractor],
        embed_model=embed_model,
        property_graph_store=graph_store
    )
    print("Fresh index created in AuraDB")

No existing graph found — initialising fresh index...
Fresh index created in AuraDB


In [46]:
#STEP 9: Process new documents and mark them as processed in AuraDB
import re
from llama_index.core import Document

def process_and_track(filename, index, driver, database):
    """Extract text, preprocess, insert into graph, mark as processed in AuraDB"""

    filepath = f"data/{filename}"
    file_type = "pdf" if filename.endswith(".pdf") else "docx"

    if file_type == "pdf":
        raw_text = extract_text_from_pdf(filepath)
    else:
        raw_text = extract_text_from_docx(filepath)

    if not raw_text.strip():
        print(f"⚠ Skipping empty document: {filename}")
        return False

    text = preprocess_text(raw_text)
    print(f"  {len(raw_text)} chars → {len(text)} chars after preprocessing")

    match = re.search(r"[Ll]ecture(\d+)", filename)
    week_label = f"Week {match.group(1)}" if match else None

    # Pre-inject WEEK anchor node
    if week_label:
        with driver.session(database=database) as session:
            session.run("""
                MERGE (l:__Node__:__Entity__:WEEK {id: $label})
                SET l.name = $label
            """, label=week_label)
        print(f"  ✓ Pre-injected {week_label} node")

    # Snapshot existing concept IDs BEFORE insertion
    with driver.session(database=database) as session:
        result = session.run("MATCH (c:CONCEPT) RETURN c.id AS id")
        existing_concept_ids = [record["id"] for record in result]

    doc = Document(
        text=text,
        metadata={"filename": filename, "file_type": file_type}
    )
    index.insert(doc)

    # Remove Chunk nodes (LlamaIndex internals)
    with driver.session(database=database) as session:
        result = session.run("MATCH (c:Chunk) DETACH DELETE c RETURN count(c) AS deleted")
        print(f"  ✓ Removed {result.single()['deleted']} Chunk nodes")

    # Wire NEW concepts (not in snapshot, not yet introduced anywhere) to this lecture
    if week_label:
        with driver.session(database=database) as session:
            result = session.run("""
                MATCH (c:CONCEPT), (l:WEEK {name: $label})
                WHERE NOT c.id IN $existing_ids
                  AND NOT (c)-[:INTRODUCED_IN]->(:WEEK)
                MERGE (c)-[:INTRODUCED_IN]->(l)
                RETURN count(c) AS wired
            """, label=week_label, existing_ids=existing_concept_ids)
            print(f"  ✓ Wired {result.single()['wired']} concepts to {week_label}")

    # Wire orphan TOPICs to this lecture
    if week_label:
        with driver.session(database=database) as session:
            result = session.run("""
                MATCH (t:TOPIC), (l:WEEK {name: $label})
                WHERE NOT (t)-[:PART_OF]->(:WEEK)
                  AND NOT t.id IN $existing_ids
                MERGE (t)-[:PART_OF]->(l)
                RETURN count(t) AS wired
            """, label=week_label, existing_ids=existing_concept_ids)
            print(f"  ✓ Wired {result.single()['wired']} topics to {week_label}")

    # Mark as processed
    with driver.session(database=database) as session:
        session.run("""
            MERGE (p:ProcessedDocument {filename: $filename})
            SET p.file_type = $file_type,
                p.processed_at = datetime()
        """, filename=filename, file_type=file_type)

    return True

if new_files:
    driver = GraphDatabase.driver(uri, auth=(username, password))
    for filename in new_files:
        print(f"\nProcessing: {filename}")
        success = process_and_track(filename, index, driver, database)
        if success:
            print(f"✓ Done: {filename}")
    driver.close()
    print(f"\nAll {len(new_files)} new documents processed!")
else:
    print("No new documents to process — graph is up to date!")

print(f"\n── Token Usage ──")
print(f"  Input tokens:  {token_counter.prompt_llm_token_count:,}")
print(f"  Output tokens: {token_counter.completion_llm_token_count:,}")
print(f"  Total:         {token_counter.total_llm_token_count:,}")


Processing: MathsComp_Lecture1.pdf
  18667 chars → 18436 chars after preprocessing
  ✓ Pre-injected Week 1 node
  ✓ Removed 6 Chunk nodes
  ✓ Wired 43 concepts to Week 1
  ✓ Wired 7 topics to Week 1
✓ Done: MathsComp_Lecture1.pdf

Processing: MathsComp_Lecture10.pdf
  8939 chars → 8804 chars after preprocessing
  ✓ Pre-injected Week 10 node
  ✓ Removed 3 Chunk nodes
  ✓ Wired 12 concepts to Week 10
  ✓ Wired 1 topics to Week 10
✓ Done: MathsComp_Lecture10.pdf

Processing: MathsComp_Lecture11.pdf
  7742 chars → 7728 chars after preprocessing
  ✓ Pre-injected Week 11 node
  ✓ Removed 2 Chunk nodes
  ✓ Wired 5 concepts to Week 11
  ✓ Wired 1 topics to Week 11
✓ Done: MathsComp_Lecture11.pdf

Processing: MathsComp_Lecture2.pdf
  10920 chars → 10631 chars after preprocessing
  ✓ Pre-injected Week 2 node
  ✓ Removed 5 Chunk nodes
  ✓ Wired 22 concepts to Week 2
  ✓ Wired 1 topics to Week 2
✓ Done: MathsComp_Lecture2.pdf

Processing: MathsComp_Lecture3.pdf
  15637 chars → 15607 chars after p

In [35]:
# STEP 10: Verify final state of AuraDB
driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session(database=database) as session:
    node_count = session.run(
        "MATCH (n) WHERE NOT n:ProcessedDocument RETURN count(n) AS count"
    ).single()["count"]
    rel_count = session.run(
        "MATCH ()-[r]->() RETURN count(r) AS count"
    ).single()["count"]
    processed_count = session.run(
        "MATCH (p:ProcessedDocument) RETURN count(p) AS count"
    ).single()["count"]

driver.close()

print(f"Knowledge graph nodes: {node_count}")
print(f"Relationships: {rel_count}")
print(f"Documents tracked: {processed_count}")

Knowledge graph nodes: 561
Relationships: 371
Documents tracked: 10


In [55]:
# STEP 11: Ontology validation — check extracted graph against ontology_v2.owl
import re
from rdflib import Graph as RDFGraph, RDF, RDFS, OWL, URIRef
from neo4j import GraphDatabase

g = RDFGraph()
g.parse("ontology_v2.owl", format="xml")

# Extract Topic individuals
ontology_topics = set()
for s, p, o in g.triples((None, RDF.type, None)):
    if str(o).split("#")[-1] == "Topic":
        labels = list(g.objects(s, RDFS.label))
        if labels:
            ontology_topics.add(str(labels[0]))
        else:
            ontology_topics.add(str(s).split("#")[-1].replace("_", " "))

# Extract Week individuals and normalise Week1 → Week 1
ontology_weeks = set()
for s, p, o in g.triples((None, RDFS.label, None)):
    label = str(o)
    if re.match(r'^Week \d+$', label):
        ontology_weeks.add(label)

# Extract Assessment individuals
ontology_assessments = set()
for s, p, o in g.triples((None, RDF.type, None)):
    type_local = str(o).split("#")[-1]
    if type_local in ["Exam", "InClassTest", "MockTest"]:
        labels = list(g.objects(s, RDFS.label))
        if labels:
            ontology_assessments.add(str(labels[0]))
        else:
            ontology_assessments.add(str(s).split("#")[-1].replace("_", " "))

# Extract week→topic mapping
HAS_TOPIC = URIRef("http://www.example.org/4COSC002W#hasTopic")
ontology_week_topics = {}
for week_uri, _, topic_uri in g.triples((None, HAS_TOPIC, None)):
    week_labels = list(g.objects(week_uri, RDFS.label))
    topic_labels = list(g.objects(topic_uri, RDFS.label))
    if week_labels and topic_labels:
        week_name = str(week_labels[0])
        topic_name = str(topic_labels[0])
        if re.match(r'^Week \d+$', week_name):
            ontology_week_topics.setdefault(week_name, set()).add(topic_name)

#  QUERY GRAPH
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session(database=database) as session:

    result = session.run("MATCH (t:TOPIC) RETURN t.name AS name")
    extracted_topics = {r["name"] for r in result}

    result = session.run("MATCH (w:WEEK) RETURN w.name AS name")
    extracted_weeks = {r["name"] for r in result}

    result = session.run("MATCH (a:ASSESSMENT) RETURN a.name AS name")
    extracted_assessments = {r["name"] for r in result}

    result = session.run("""
        MATCH (c:CONCEPT)
        WHERE NOT (c)-[:INTRODUCED_IN]->(:WEEK)
          AND NOT (c)-[:PART_OF]->(:WEEK)
        RETURN c.name AS name
    """)
    orphan_concepts = [r["name"] for r in result]

    result = session.run("""
        MATCH (w:WEEK)
        WHERE NOT (w)<-[:INTRODUCED_IN]-(:CONCEPT)
        RETURN w.name AS name
    """)
    empty_weeks = [r["name"] for r in result if r["name"] not in {"Week 6"}]

    total_concepts = session.run(
        "MATCH (c:CONCEPT) RETURN count(c) AS n").single()["n"]
    anchored_concepts = session.run("""
        MATCH (c:CONCEPT)
        WHERE (c)-[:INTRODUCED_IN]->(:WEEK)
        RETURN count(c) AS n""").single()["n"]
    concepts_with_def = session.run("""
        MATCH (c:CONCEPT)
        WHERE (c)-[:HAS_DEFINITION]->(:DEFINITION)
        RETURN count(c) AS n""").single()["n"]
    used_rel_types = session.run("""
        MATCH ()-[r]->()
        RETURN count(DISTINCT type(r)) AS n""").single()["n"]
    duplicate_concepts = session.run("""
        MATCH (c:CONCEPT)
        WITH toLower(c.name) AS name, count(*) AS cnt
        WHERE cnt > 1
        RETURN count(*) AS n""").single()["n"]
    noise_row = session.run("""
        MATCH (c:CONCEPT)
        RETURN sum(CASE WHEN size(c.name) > 50 THEN 1 ELSE 0 END) AS n,
               count(c) AS total""").single()

driver.close()

#  VALIDATION REPORT 
print("\n" + "="*55)
print("ONTOLOGY VALIDATION REPORT")
print("="*55)

invalid_topics = extracted_topics - ontology_topics
missing_topics = ontology_topics - extracted_topics
invalid_weeks = {w for w in extracted_weeks if not re.match(r'Week \d+$', w)}

print(f"\n✓ Valid topics extracted: {len(extracted_topics - invalid_topics)}")
if invalid_topics:
    print(f"✗ Invalid topics (not in ontology): {sorted(invalid_topics)}")
if missing_topics:
    processed_weeks = len(extracted_weeks) - len(invalid_weeks)
    print(f"⚠ Topics not yet extracted ({len(missing_topics)}) — "
          f"{processed_weeks}/{len(ontology_weeks)} weeks processed:")
    print(f"  {sorted(missing_topics)}")

if invalid_weeks:
    print(f"\n✗ Invalid WEEK nodes: {sorted(invalid_weeks)}")
else:
    print(f"\n✓ All WEEK nodes correctly named")

ASSESSMENT_NAME_MAP = {
    "ICT": "In-Class Test (ICT)",
    "Exam": "Final Examination",
    "Mock Test": "Mock Test (Week 7)"
}
normalised_extracted = {ASSESSMENT_NAME_MAP.get(a, a) for a in extracted_assessments}
invalid_assessments = normalised_extracted - ontology_assessments
missing_assessments = ontology_assessments - normalised_extracted

if not invalid_assessments and not missing_assessments:
    print(f"\n✓ Assessments valid: {sorted(normalised_extracted)}")
else:
    if invalid_assessments:
        print(f"\n✗ Invalid assessments: {sorted(invalid_assessments)}")
    if missing_assessments:
        print(f"\n⚠ Assessments expected but missing: {sorted(missing_assessments)}")

if orphan_concepts:
    print(f"\n⚠ Concepts with no WEEK anchor ({len(orphan_concepts)}): {orphan_concepts}")
else:
    print(f"\n✓ All concepts anchored to a WEEK")

if empty_weeks:
    print(f"\n⚠ WEEK nodes with no concepts: {empty_weeks}")
else:
    print(f"\n✓ All WEEK nodes have concepts")

print(f"\n── Week-Topic Alignment ──")
driver = GraphDatabase.driver(uri, auth=(username, password))
correct_alignments = 0
total_alignments = 0
with driver.session(database=database) as session:
    result = session.run("""
        MATCH (t:TOPIC)-[:PART_OF]->(w:WEEK)
        RETURN w.name AS week, collect(t.name) AS topics
        ORDER BY week
    """)
    for record in result:
        week = record["week"]
        actual = set(record["topics"])
        expected = ontology_week_topics.get(week, set())
        if expected:
            correct_alignments += len(actual & expected)
            total_alignments += len(expected)
        wrong = actual - expected
        missing = expected - actual
        if not wrong and not missing:
            print(f"  ✓ {week}: topics correct")
        else:
            if wrong:
                print(f"  ✗ {week}: wrong topics {sorted(wrong)}")
            if missing:
                print(f"  ⚠ {week}: missing topics {sorted(missing)}")
driver.close()

#  ONTOLOGY CONFORMANCE SCORES 
# These metrics compare directly against ontology_v2.owl ground truth
print("\n" + "="*55)
print("ONTOLOGY CONFORMANCE SCORES")
print("(comparing extracted graph against ontology_v2.owl)")
print("="*55)

tp = len(extracted_topics & ontology_topics)
fp = len(extracted_topics - ontology_topics)
fn = len(ontology_topics - extracted_topics)
topic_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
topic_recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
topic_f1        = (2 * topic_precision * topic_recall /
                  (topic_precision + topic_recall)
                  if (topic_precision + topic_recall) > 0 else 0)

week_coverage   = (len(extracted_weeks) - len(invalid_weeks)) / len(ontology_weeks) \
                  if ontology_weeks else 0
alignment_score = correct_alignments / total_alignments \
                  if total_alignments > 0 else 0

print(f"\nTopic extraction (vs ontology topic list)")
print(f"  Precision:            {topic_precision:.1%}")
print(f"  Recall:               {topic_recall:.1%}")
print(f"  F1:                   {topic_f1:.1%}")
print(f"\nWeek coverage (vs ontology week list)")
print(f"  {len(extracted_weeks) - len(invalid_weeks)}/{len(ontology_weeks)} weeks correctly named: {week_coverage:.1%}")
print(f"\nWeek-topic alignment (vs ontology hasTopic)")
print(f"  {correct_alignments}/{total_alignments} topic-week pairs correct: {alignment_score:.1%}")

#  GRAPH QUALITY METRICS 
# These measure internal pipeline quality — not validated against the ontology
print("\n" + "="*55)
print("GRAPH QUALITY METRICS")
print("(internal pipeline assessment — not ontology-derived)")
print("="*55)

anchor_rate    = anchored_concepts / total_concepts if total_concepts else 0
def_coverage   = concepts_with_def / total_concepts if total_concepts else 0
schema_rel_types = 9
rel_diversity  = used_rel_types / schema_rel_types
noise_rate     = noise_row["n"] / noise_row["total"] if noise_row["total"] > 0 else 0

print(f"\n  Concept anchor rate:    {anchor_rate:.1%}  ({anchored_concepts}/{total_concepts} concepts linked to a WEEK)")
print(f"  Definition coverage:    {def_coverage:.1%}  ({concepts_with_def}/{total_concepts} concepts have HAS_DEFINITION)")
print(f"  Relationship diversity: {rel_diversity:.1%}  ({used_rel_types}/{schema_rel_types} schema relation types in use)")
print(f"  Duplicate concepts:     {duplicate_concepts} duplicates detected")
print(f"  Noise rate:             {noise_rate:.1%}  (concepts with name > 50 chars)")
print("="*55)


ONTOLOGY VALIDATION REPORT

✓ Valid topics extracted: 8
✗ Invalid topics (not in ontology): ['Discrete Mathematics']
⚠ Topics not yet extracted (7) — 10/12 weeks processed:
  ['Exam Revision', 'Intervals', 'Module Introduction', 'Relations', 'Sequences', 'Set Theory', 'Statistics Basics']

✓ All WEEK nodes correctly named

⚠ Assessments expected but missing: ['Final Examination', 'In-Class Test (ICT)', 'Mock Test (Week 11)', 'Mock Test (Week 7)']

✓ All concepts anchored to a WEEK

✓ All WEEK nodes have concepts

── Week-Topic Alignment ──
  ✗ Week 1: wrong topics ['Discrete Mathematics', 'Functions', 'Graph Theory', 'Logics', 'Matrices Part 1', 'Probability']
  ⚠ Week 1: missing topics ['Module Introduction', 'Sequences']
  ✓ Week 11: topics correct
  ✓ Week 8: topics correct

ONTOLOGY CONFORMANCE SCORES
(comparing extracted graph against ontology_v2.owl)

Topic extraction (vs ontology topic list)
  Precision:            88.9%
  Recall:               53.3%
  F1:                   66.

In [ ]:
# STEP 12: Natural language query engine
import nest_asyncio
nest_asyncio.apply()

from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

query_engine = index.as_query_engine(
    llm=llm,
    include_text=True,
    response_mode="tree_summarize"
)

print("Query engine ready!")

def ask(question):
    response = query_engine.query(question)
    print(f"Q: {question}")
    print(f"A: {response}")
    print("-" * 50)

ask("What concepts are introduced in Week 1?")

Query engine ready!
Q: What concepts are introduced in Week 1?
A: Ontologies, Algorithm, Project Network Diagram, and Elementary Logics.
--------------------------------------------------
